# AITR Lab 02 - Negative-flip-aware model updates (Colab notebook)
Pathway B: controlled comparison of **NAIVE retraining**, a **PCT-style consistency penalty** and **BCWI weight interpolation** on sklearn *digits*.

How to use: *Runtime -> Run all* (CPU is enough, about 2 minutes). Every cell is self-contained; no files need uploading.

**Honest notes**
- The PCT-style penalty here is a *simplified* variant, not a verified reproduction of Yan et al. (2021).
- Hyperparameters are chosen on a **validation** split; the **test** split is used only for final reporting.
- "Update magnitude" = Gaussian noise (sigma) added to the new model's training images. This is an inferred proxy, not an LLM update.

## Cell 1 - Environment check (Prompt 08)

In [ ]:
import os, time, csv, platform, sys
import numpy as np, pandas as pd, sklearn, matplotlib
import matplotlib.pyplot as plt

print("python     :", platform.python_version())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("sklearn    :", sklearn.__version__)
print("matplotlib :", matplotlib.__version__)
print("time       :", time.strftime("%Y-%m-%d %H:%M:%S"))
for d in ["results/csv", "results/figures", "results/logs"]:
    os.makedirs(d, exist_ok=True)

## Cell 2 - Frozen configuration
Change values only *before* running the experiment cells; record any change in your decision log.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
SIGMAS = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0]      # update magnitude (noise std on NEW training images)
SIGMA_ABL = 8.0                               # magnitude used for hyperparameter selection
LAMBDAS = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]     # PCT penalty weight
ALPHAS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.92, 0.95, 0.98, 1.0]  # BCWI interpolation
PCT_LAMBDA_CARRIED = 0.25                     # mildest tested lambda, carried to the sweep

TEST_FRACTION = 0.20       # fixed held-out test set (never used for selection)
VAL_FRACTION = 0.20        # of the non-test pool, used for hyperparameter selection
OLD_MODEL_FRACTION = 0.25  # OLD model trains on this fraction of the training pool
N_EPOCHS, LR, BATCH, L2 = 300, 0.5, 64, 1e-4
PIXEL_MAX = 16.0

# Selection criterion (written down BEFORE the final sweep): >=20% NFR reduction vs NAIVE
# at <=2 points accuracy cost. alpha=1.0 is excluded (it IS the old model, not an update).
CRIT_NFR_REDUCTION, CRIT_ACC_COST = 0.20, 0.02

## Cell 3 - Data utilities (dataset loads offline from scikit-learn)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

def load_full_dataset():
    d = load_digits()
    return d.data.astype(np.float64), d.target.astype(np.int64)

def make_fixed_test_split(X, y, seed):
    sp = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    pool_idx, test_idx = next(sp.split(X, y))
    return X[pool_idx], y[pool_idx], X[test_idx], y[test_idx]

def make_val_split(Xp, yp, seed):
    sp = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=seed + 7)
    tr, va = next(sp.split(Xp, yp))
    return Xp[tr], yp[tr], Xp[va], yp[va]

def subsample_pool(Xp, yp, fraction, seed):
    if fraction >= 1.0:
        order = np.random.RandomState(seed).permutation(len(yp))
        return Xp[order], yp[order]
    sp = StratifiedShuffleSplit(n_splits=1, train_size=fraction, random_state=seed)
    idx, _ = next(sp.split(Xp, yp))
    return Xp[idx], yp[idx]

def corrupt_images(X, sigma, seed):
    # additive Gaussian pixel noise on every row, clipped to the valid pixel range
    if sigma <= 0:
        return X.copy()
    rng = np.random.RandomState(seed)
    Xc = X.copy()
    n = len(X)
    idx = rng.choice(n, size=n, replace=False)
    noise = rng.normal(0, sigma, size=(n, X.shape[1]))
    Xc[idx] = np.clip(Xc[idx] + noise, 0, PIXEL_MAX)
    return Xc

## Cell 4 - Model: softmax classifier (NumPy) with PCT-style penalty and BCWI interpolation

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

class SoftmaxClassifier:
    def __init__(self, n_features, n_classes=10, lr=LR, n_epochs=N_EPOCHS, batch_size=BATCH,
                 l2=L2, seed=0, init_W=None, init_b=None):
        self.n_features, self.n_classes = n_features, n_classes
        self.lr, self.n_epochs, self.batch_size, self.l2, self.seed = lr, n_epochs, batch_size, l2, seed
        rng = np.random.RandomState(seed)
        if init_W is not None:
            self.W, self.b = init_W.copy(), init_b.copy()
        else:
            self.W, self.b = rng.normal(0, 0.01, (n_features, n_classes)), np.zeros(n_classes)

    def predict_proba(self, X): return softmax(X @ self.W + self.b)
    def predict(self, X): return self.predict_proba(X).argmax(axis=1)

    def fit(self, X, y, old_correct_mask=None, old_pred=None, pct_lambda=0.0):
        n = len(y)
        rng = np.random.RandomState(self.seed + 1000)
        Y = np.zeros((n, self.n_classes)); Y[np.arange(n), y] = 1.0
        if pct_lambda > 0:
            T = np.zeros((n, self.n_classes)); T[np.arange(n), old_pred] = 1.0
            M = old_correct_mask.astype(np.float64).reshape(-1, 1)
        history = []
        n_batches = max(1, n // self.batch_size)
        for _ in range(self.n_epochs):
            order = rng.permutation(n); loss = 0.0
            for b in np.array_split(order, n_batches):
                Xb, Yb = X[b], Y[b]
                P = softmax(Xb @ self.W + self.b)
                g = (P - Yb) / len(b)
                if pct_lambda > 0:
                    Mb = M[b]
                    g = g + pct_lambda * Mb * (P - T[b]) / max(1.0, Mb.sum())
                self.W -= self.lr * (Xb.T @ g + self.l2 * self.W)
                self.b -= self.lr * g.sum(axis=0)
                loss += -(Yb * np.log(P + 1e-12)).sum(axis=1).mean()
            history.append(loss / n_batches)
        return history

def interpolate_weights(old, new, alpha):
    # BCWI-style: alpha=0 -> new model, alpha=1 -> old model
    m = SoftmaxClassifier(new.n_features, new.n_classes, seed=new.seed,
                          init_W=alpha * old.W + (1 - alpha) * new.W,
                          init_b=alpha * old.b + (1 - alpha) * new.b)
    return m

## Cell 5 - Metrics and experiment harness
NFR = share of examples the old model got right that the new model gets wrong (Bansal et al., 2019).

In [ ]:
def accuracy(y, p): return float((y == p).mean())
def negative_flip_rate(y, old, new): return float(((old == y) & (new != y)).sum() / len(y))
def positive_flip_rate(y, old, new): return float(((old != y) & (new == y)).sum() / len(y))
def negative_flip_indices(y, old, new): return np.where((old == y) & (new != y))[0]

def build_condition(seed, sigma):
    X, y = load_full_dataset()
    Xp_full, yp_full, X_test, y_test = make_fixed_test_split(X, y, seed)
    Xp, yp, X_val, y_val = make_val_split(Xp_full, yp_full, seed)
    X_old, y_old = subsample_pool(Xp, yp, OLD_MODEL_FRACTION, seed)
    X_new_clean, y_new = subsample_pool(Xp, yp, 1.0, seed)
    X_new = corrupt_images(X_new_clean, sigma, seed)
    sc = StandardScaler().fit(X_new_clean)                      # fit on training data only
    rng = np.random.RandomState(seed)
    W0, b0 = rng.normal(0, 0.01, (64, 10)), np.zeros(10)        # shared init (needed for BCWI)
    old = SoftmaxClassifier(64, seed=seed, init_W=W0, init_b=b0)
    old.fit(sc.transform(X_old), y_old)
    return dict(X_new_s=sc.transform(X_new), y_new=y_new,
                X_test_s=sc.transform(X_test), y_test=y_test,
                X_val_s=sc.transform(X_val), y_val=y_val,
                old=old, W0=W0, b0=b0, seed=seed, sigma=sigma)

def train_naive(c):
    m = SoftmaxClassifier(64, seed=c["seed"], init_W=c["W0"], init_b=c["b0"])
    m.fit(c["X_new_s"], c["y_new"]); return m

def train_pct(c, lam):
    op = c["old"].predict(c["X_new_s"])
    m = SoftmaxClassifier(64, seed=c["seed"], init_W=c["W0"], init_b=c["b0"])
    m.fit(c["X_new_s"], c["y_new"], old_correct_mask=(op == c["y_new"]), old_pred=op, pct_lambda=lam)
    return m

def evaluate(c, model, label, split="test"):
    assert split in ("val", "test")
    X, y = c[f"X_{split}_s"], c[f"y_{split}"]
    po, pn = c["old"].predict(X), model.predict(X)
    return dict(label=label, seed=c["seed"], sigma=c["sigma"], accuracy=accuracy(y, pn),
                nfr=negative_flip_rate(y, po, pn), pfr=positive_flip_rate(y, po, pn),
                acc_old=accuracy(y, po), split=split)
print("harness ready")

## Cell 6 - Data sanity check (Gate G2): shapes, determinism, no train/test overlap

In [ ]:
X, y = load_full_dataset()
print("X", X.shape, "y", y.shape, "classes", sorted(set(y.tolist())))
assert X.shape == (1797, 64) and len(set(y.tolist())) == 10

sp = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=42)
a = next(sp.split(X, y)); b = next(StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=42).split(X, y))
assert np.array_equal(a[1], b[1]), "split not deterministic"
assert len(set(a[0]) & set(a[1])) == 0, "train/test overlap"
print("PASS: split is deterministic and train/test are disjoint; test size =", len(a[1]))

## Cell 7 - Baseline validation (single reference seed): does the pipeline behave sensibly?
Diagnostic only - not a reported result.

In [ ]:
c = build_condition(seed=42, sigma=0.0)
naive = train_naive(c)
r = evaluate(c, naive, "NAIVE", split="val")
print(f"OLD   val acc = {r['acc_old']:.4f}")
print(f"NAIVE val acc = {r['accuracy']:.4f}   NFR = {r['nfr']:.4f}   PFR = {r['pfr']:.4f}")
hist = SoftmaxClassifier(64, seed=0).fit(c["X_new_s"], c["y_new"])
print(f"training loss: first epoch {hist[0]:.3f} -> last {hist[-1]:.4f}")
assert hist[-1] < hist[0], "loss did not decrease"
assert r["accuracy"] > 0.5, "implausibly low accuracy"
print("PASS: model trains, accuracy plausible, NFR measured.")

## Cell 8 - Design pilot on the VALIDATION split (why sigma is the magnitude axis)

In [ ]:
rows = []
for sigma in SIGMAS:
    nfrs, accs = [], []
    for s in SEEDS:
        c = build_condition(s, sigma); r = evaluate(c, train_naive(c), "NAIVE", split="val")
        nfrs.append(r["nfr"]); accs.append(r["accuracy"])
    rows.append((sigma, np.mean(accs), np.mean(nfrs)))
pilot = pd.DataFrame(rows, columns=["sigma", "NAIVE_val_acc", "NAIVE_val_NFR"]).round(4)
print(pilot.to_string(index=False))
pilot.to_csv("results/csv/pilot_sigma_validation.csv", index=False)

## Cell 9 - Ablation on the VALIDATION split (sigma = 8): sweep PCT lambda and BCWI alpha

In [ ]:
abl = []
for s in SEEDS:
    c = build_condition(s, SIGMA_ABL)
    naive = train_naive(c)
    abl.append(evaluate(c, naive, "NAIVE", split="val"))
    for lam in LAMBDAS:
        m = naive if lam == 0.0 else train_pct(c, lam)
        abl.append(evaluate(c, m, f"PCT_lambda={lam}", split="val"))
    for al in ALPHAS:
        m = naive if al == 0.0 else interpolate_weights(c["old"], naive, al)
        abl.append(evaluate(c, m, f"BCWI_alpha={al}", split="val"))
abl = pd.DataFrame(abl); abl.to_csv("results/csv/ablation_results.csv", index=False)
print(len(abl), "rows saved (all on the validation split)")

## Cell 10 - Select lambda* / alpha* with the frozen criterion (validation only)

In [ ]:
nv = abl[abl.label == "NAIVE"]; n_acc, n_nfr = nv.accuracy.mean(), nv.nfr.mean()
print(f"NAIVE val: acc={n_acc:.4f} NFR={n_nfr:.4f} | OLD val acc={nv.acc_old.mean():.4f}\n")

def scan(prefix, grid, exclude=None):
    best = None
    for g in grid:
        sub = abl[abl.label == f"{prefix}={g}"]; a, f = sub.accuracy.mean(), sub.nfr.mean()
        red, cost = 1 - f / n_nfr, n_acc - a
        ok = (red >= CRIT_NFR_REDUCTION) and (cost <= CRIT_ACC_COST) and (g != exclude)
        print(f"{prefix}={g:<5} acc={a:.4f} NFR={f:.4f} NFR_reduction={red:+.1%} acc_cost={cost:+.4f}" + ("  <= MEETS CRITERION" if ok else ""))
        if ok and (best is None or f < best[1]): best = (g, f)
    return None if best is None else best[0]

print("--- PCT ---");  LAMBDA_STAR = scan("PCT_lambda", LAMBDAS)
print("\n--- BCWI (alpha=1.0 excluded: structural identity = old model) ---"); ALPHA_STAR = scan("BCWI_alpha", ALPHAS, exclude=1.0)
print(f"\nSELECTED: lambda* = {LAMBDA_STAR}   alpha* = {ALPHA_STAR}")
assert ALPHA_STAR is not None

## Cell 11 - Frozen comparison on the TEST split (all sigma, 5 seeds)
`alpha*` comes from Cell 10; PCT uses the mildest lambda. No re-tuning per sigma.

In [ ]:
res = []
for s in SEEDS:
    for sigma in SIGMAS:
        c = build_condition(s, sigma)
        naive = train_naive(c)
        res.append(evaluate(c, c["old"], "OLD_frozen"))
        res.append(evaluate(c, naive, "NAIVE"))
        res.append(evaluate(c, train_pct(c, PCT_LAMBDA_CARRIED), "PCT"))
        res.append(evaluate(c, interpolate_weights(c["old"], naive, ALPHA_STAR), "BCWI"))
res = pd.DataFrame(res); assert (res.split == "test").all()
res.to_csv("results/csv/magnitude_sweep_results.csv", index=False)
summary = res.groupby(["label", "sigma"]).agg(acc_mean=("accuracy","mean"), acc_std=("accuracy","std"),
          nfr_mean=("nfr","mean"), nfr_std=("nfr","std"), pfr_mean=("pfr","mean")).reset_index()
summary.to_csv("results/csv/magnitude_sweep_summary.csv", index=False)
print(summary.round(4).to_string(index=False))

## Cell 12 - Result tables and per-seed consistency check (no significance test is claimed)

In [ ]:
for metric in ["nfr", "accuracy"]:
    t = res.pivot_table(index="sigma", columns="label", values=metric, aggfunc=["mean", "std"])
    print(f"\n=== test {metric}: mean (std) ===")
    out = pd.DataFrame({m: [f"{t['mean'][m][s]:.3f} ({t['std'][m][s]:.3f})" for s in t.index]
                        for m in ["OLD_frozen", "NAIVE", "PCT", "BCWI"]}, index=t.index)
    print(out.to_string())
print("\n=== seeds (of 5) where method NFR < NAIVE NFR ===")
chk = []
for sigma in SIGMAS:
    p = res[res.sigma == sigma].pivot(index="seed", columns="label", values="nfr")
    chk.append((sigma, int((p.BCWI < p.NAIVE).sum()), int((p.PCT < p.NAIVE).sum())))
print(pd.DataFrame(chk, columns=["sigma", "BCWI<NAIVE", "PCT<NAIVE"]).to_string(index=False))

## Cell 13 - Deeper analysis: which classes carry the negative flips? (sigma = 8, test)

In [ ]:
from collections import Counter
cn, cb = Counter(), Counter(); tn = tb = 0
for s in SEEDS:
    c = build_condition(s, 8.0); naive = train_naive(c); bc = interpolate_weights(c["old"], naive, ALPHA_STAR)
    y_t, Xt = c["y_test"], c["X_test_s"]; po = c["old"].predict(Xt)
    fn = negative_flip_indices(y_t, po, naive.predict(Xt)); fb = negative_flip_indices(y_t, po, bc.predict(Xt))
    tn += len(fn); tb += len(fb)
    cn.update(int(y_t[i]) for i in fn); cb.update(int(y_t[i]) for i in fb)
print(f"total negative flips over 5 seeds: NAIVE={tn}  BCWI={tb}  (reduction {1 - tb / tn:+.1%})")
byclass = pd.DataFrame({"digit": range(10), "NAIVE": [cn.get(k, 0) for k in range(10)], "BCWI": [cb.get(k, 0) for k in range(10)]})
print(byclass.to_string(index=False)); byclass.to_csv("results/csv/failure_analysis_by_class.csv", index=False)

## Cell 14 - Figures (generated from the DataFrames above; saved to results/figures)

In [ ]:
COL = {"OLD_frozen": "#888888", "NAIVE": "#d62728", "PCT": "#ff7f0e", "BCWI": "#1f77b4"}
MK = {"OLD_frozen": "s", "NAIVE": "o", "PCT": "^", "BCWI": "D"}

# Fig 1: validation trade-off (ablation)
fig, ax = plt.subplots(figsize=(5.4, 4.2))
for pref, name, col, mk in [("PCT_lambda", "PCT (lambda sweep)", COL["PCT"], "^"), ("BCWI_alpha", "BCWI (alpha sweep)", COL["BCWI"], "D")]:
    sub = abl[abl.label.str.startswith(pref + "=")].copy(); sub["g"] = sub.label.str.split("=").str[1].astype(float)
    g = sub.groupby("g").agg(a=("accuracy","mean"), f=("nfr","mean")).sort_index()
    ax.plot(g.f, g.a, marker=mk, color=col, label=name)
ax.scatter([n_nfr], [n_acc], color=COL["NAIVE"], s=70, zorder=5, label="NAIVE")
ax.set_xlabel("Negative flip rate (lower is better)"); ax.set_ylabel("Accuracy (higher is better)")
ax.set_title("Validation trade-off at sigma=8"); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.savefig("results/figures/fig1_tradeoff_ablation.png", dpi=200, bbox_inches="tight"); plt.show()

# Fig 2 / 3: NFR and accuracy vs sigma (test)
for metric, ylab, fname in [("nfr", "Negative flip rate", "fig2_nfr_vs_sigma"), ("accuracy", "Test accuracy", "fig3_accuracy_vs_sigma")]:
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    for m in ["OLD_frozen", "NAIVE", "PCT", "BCWI"]:
        s = summary[summary.label == m].sort_values("sigma"); k = "nfr" if metric == "nfr" else "acc"
        ax.plot(s.sigma, s[f"{k}_mean"], marker=MK[m], color=COL[m], label=m)
        ax.fill_between(s.sigma, s[f"{k}_mean"] - s[f"{k}_std"], s[f"{k}_mean"] + s[f"{k}_std"], color=COL[m], alpha=.15)
    ax.set_xlabel("Update magnitude sigma"); ax.set_ylabel(ylab); ax.legend(fontsize=8); ax.grid(alpha=.3)
    ax.set_title(f"{ylab} vs sigma (mean +/- std, 5 seeds)")
    fig.savefig(f"results/figures/{fname}.png", dpi=200, bbox_inches="tight"); plt.show()

# Fig 4: per-class negative flips
fig, ax = plt.subplots(figsize=(5.4, 3.6)); x = np.arange(10)
ax.bar(x - .18, byclass.NAIVE, .36, color=COL["NAIVE"], label="NAIVE"); ax.bar(x + .18, byclass.BCWI, .36, color=COL["BCWI"], label="BCWI")
ax.set_xticks(x); ax.set_xlabel("True digit class"); ax.set_ylabel("Negative flips (5 seeds, sigma=8)"); ax.legend(fontsize=8)
fig.savefig("results/figures/fig4_failure_by_class.png", dpi=200, bbox_inches="tight"); plt.show()

## Cell 15 - Save everything and download (works in Colab; elsewhere it just zips)

In [ ]:
import shutil
shutil.make_archive("aitr_lab02_results", "zip", "results")
print("created aitr_lab02_results.zip with:", sorted(os.listdir("results/csv")), sorted(os.listdir("results/figures")))
try:
    from google.colab import files
    files.download("aitr_lab02_results.zip")
except ImportError:
    print("(not running in Colab - download skipped)")

## Cell 16 - Your AI-assistance log (Prompt 18) - write this yourself
Record only what really happened. Suggested rows: *task - AI proposal - what you verified/changed - outcome*.

| # | Task | AI proposal | Your verification / correction | Outcome |
|---|---|---|---|---|
| 1 | [STUDENT TO COMPLETE] | | | |

Also list at least three REAL AI limitations or corrections you observed, which conclusions rest on your own reasoning, and the risks that remain.